# Notebook 13 — Subagents for the Research Deep Agent

Subagents let the main research agent delegate focused multi-step work into narrower contexts.

## Core mental model

```text
Single agent:
one context does everything

Deep Agent + subagents:
main agent delegates focused work
        ↓
specialist works in narrower context
        ↓
returns concise findings
        ↓
main agent synthesizes
```

Subagents are primarily a **specialization + context-partitioning strategy**.

## Learning goals

1. What a subagent is
2. Why subagents help research
3. Parent vs specialist responsibilities
4. Declarative subagent specs
5. Context isolation
6. Subagents vs Skills vs Tools
7. Delegation/routing
8. Isolation vs inherited context
9. Sequential vs parallel delegation
10. Failure modes and over-agenting
11. Delegation evaluation
12. Economics: parent-context efficiency vs total system work

# 13.1 — What is a subagent?

A tool performs one action:

```text
web_search(query)
```

A subagent performs a multi-step mini-task:

```text
"Research the identity and RBAC model of Foundry Hosted Agents."
  ↓
search
  ↓
reason
  ↓
possibly use more tools
  ↓
return focused synthesis
```

# 13.2 — Why subagents help research

A broad request may require several perspectives:

```text
Main Research Agent
   ├── architecture researcher
   ├── identity researcher
   └── economics researcher
          ↓
      focused findings
          ↓
    parent synthesis
```

The parent acts like a research lead.

# 13.3 — Context isolation

A specialist usually does not need the full parent conversation.

```text
Parent context: large
Delegated task: focused
Subagent context: only what the specialist needs
```

This can reduce context pollution.

But remember:

```text
main-context efficiency
≠
total token efficiency
```

# 13.4 — Inspect the installed Deep Agents subagent API

In [ ]:
import inspect
from deepagents import create_deep_agent

print(inspect.signature(create_deep_agent))

try:
    import deepagents.middleware.subagents as sm
    print(sorted(n for n in dir(sm) if not n.startswith("_")))
except Exception as exc:
    print("Could not inspect subagents module:", exc)

# 13.5 — Reuse the real research-agent components

In [ ]:
from deep_agents_foundry.agent import RESEARCH_INSTRUCTIONS
from deep_agents_foundry.model import build_model
from deep_agents_foundry.tools import build_web_search_tool
from deep_agents_foundry import build_sqlite_checkpointer, thread_config, content_text

model = build_model()
web_search = build_web_search_tool()
checkpointer = build_sqlite_checkpointer("../data/notebook_13_subagents.db")

# 13.6 — Load reusable subagent specs

In [ ]:
from pathlib import Path
import sys

root = Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from subagents.architecture_researcher import ARCHITECTURE_RESEARCHER
from subagents.identity_researcher import IDENTITY_RESEARCHER
from subagents.economics_researcher import ECONOMICS_RESEARCHER

architecture_researcher = {**ARCHITECTURE_RESEARCHER, "tools": [web_search]}
identity_researcher = {**IDENTITY_RESEARCHER, "tools": [web_search]}
economics_researcher = {**ECONOMICS_RESEARCHER, "tools": [web_search]}

# 13.7 — First subagent experiment

Start with one specialist only.

```text
Main Research Agent
   ↓ task
Architecture Researcher
   ↓
focused synthesis
```

In [ ]:
# After verifying the installed subagents= contract:

# architecture_agent = create_deep_agent(
#     model=model,
#     tools=[web_search],
#     system_prompt=RESEARCH_INSTRUCTIONS,
#     subagents=[architecture_researcher],
#     checkpointer=checkpointer,
# )

In [ ]:
# Example task:
#
# result = architecture_agent.invoke(
#     {"messages": [{
#         "role": "user",
#         "content": (
#             "Research Microsoft Foundry Hosted Agents. Focus on architecture, "
#             "state/session boundaries, deployment model, scalability, and trade-offs."
#         )
#     }]},
#     config=thread_config("subagent-architecture-1"),
# )
# print(content_text(result["messages"][-1]))

# 13.8 — Add a second specialist

Now routing matters:

```text
"Research Hosted Agent authentication and RBAC."
→ identity-researcher

"Explain runtime, scaling, and state boundaries."
→ architecture-researcher

"What is a Hosted Agent?"
→ ideally parent answers directly
```

Good agents do not delegate merely because subagents exist.

In [ ]:
# two_subagent_agent = create_deep_agent(
#     model=model,
#     tools=[web_search],
#     system_prompt=RESEARCH_INSTRUCTIONS,
#     subagents=[architecture_researcher, identity_researcher],
#     checkpointer=checkpointer,
# )

# 13.9 — Why descriptions matter

The parent routes using specialist metadata.

Bad:

```text
"Helps with research."
```

Good:

```text
"Research authentication, authorization, managed identity, RBAC,
trust boundaries, and credential flows."
```

# 13.10 — Subagents vs Skills vs Tools

```text
Skill
= procedural knowledge

Subagent
= execution delegation

Tool
= primitive capability
```

Example:

```text
Skill:
How to compare architectures

Subagent:
Delegate architecture research

Tool:
Search the web
```

A subagent can itself have Skills and Tools.

# 13.11 — Isolation vs inherited context

Two patterns:

```text
isolated:
parent → task description → specialist

inherited/forked:
parent conversation → specialist
```

Start with **isolated specialists** for cleaner context and easier evaluation.
Use inherited context only when genuinely necessary.

# 13.12 — Sequential vs parallel delegation

Sequential:

```text
architecture → identity → economics → synthesis
```

Parallel:

```text
          architecture
         ↗
parent ──→ identity
         ↘
          economics
```

Parallelism can reduce wall-clock latency while increasing concurrent resource usage.

# 13.13 — Main-context efficiency vs total work

Subagents may shrink what the parent sees:

```text
specialist does large research
    ↓
returns concise result
    ↓
parent sees summary only
```

But total work may increase:

```text
parent calls
+ specialist calls
+ searches/tools
+ synthesis
```

So parent-context savings do not guarantee total-token savings.

# 13.14 — Baseline vs subagent experiment

Use the same complex task in two architectures.

### Baseline
one Deep Agent

### Subagent version
main agent + architecture + identity specialists

Track:

| Metric | Baseline | Subagents |
|---|---:|---:|
| Quality | | |
| Source quality | | |
| Coverage | | |
| Input tokens | | |
| Output tokens | | |
| Model calls | | |
| Web searches | | |
| Latency | | |
| Duplicate work | | |

# 13.15 — Failure modes

- wrong delegation
- unnecessary delegation
- duplicate research
- overlapping specialists
- weak task descriptions
- too little/too much delegated context
- subagent returns too much detail
- conflicting specialist conclusions
- cascading tool calls
- more tokens without enough quality gain
- over-agenting simple tasks

## Classic over-agenting pattern

```text
simple task
   ↓
parent delegates
   ↓
specialist researches
   ↓
parent synthesizes
```

when the parent could have answered directly.

A good rule:

```text
delegate when:
task is complex
AND specialization helps
AND separate context is useful
```

# 13.16 — Evaluate delegation separately from outcome

## Delegation quality
- Should the parent delegate?
- Did it delegate?
- Did it choose the right specialist?
- Did it delegate too often?

## Outcome quality
- Did factual quality improve?
- Did coverage improve?
- Did source quality improve?
- Did synthesis improve?

Then measure economics:

```text
Δ quality
Δ tokens
Δ latency
Δ searches
```

## Small routing dataset

| Prompt | Expected |
|---|---|
| What is a Hosted Agent? | Parent direct |
| Research runtime/scaling | Architecture specialist |
| Research managed identity/RBAC | Identity specialist |
| Analyze token/cost implications | Economics specialist if complex |
| Say hello | Parent direct |

# 13.17 — Specialist output should be compact

Good specialist return:

```text
key findings
supporting evidence
important uncertainty
trade-offs
```

Poor specialist return:

```text
all search snippets
all intermediate material
all raw working context
```

The parent needs signal, not the specialist's full scratchpad.

# 13.18 — Conflicting specialists

Different specialists may surface different legitimate perspectives.

Example:

```text
Architecture:
Hosted abstraction reduces operational burden.

Economics:
Hosted abstraction can reduce cost transparency/control.
```

The parent must synthesize the trade-off rather than choose one blindly.

# 13.19 — Recommended progression

```text
Experiment 1:
architecture only

Experiment 2:
architecture + identity

Experiment 3:
economics only if evaluation justifies it
```

Do not begin with a committee of agents.

# 13.20 — Relationship to Memory and Skills

```text
Memory
→ retrieve relevant durable knowledge/preferences

Skills
→ retrieve relevant procedure

Subagents
→ delegate focused execution into separate context
```

Decision intuition:

```text
same context is enough?
→ memory / skills / tools

separate focused context helps?
→ consider subagent
```

# 13.21 — Economics mental model

```text
Total work
=
parent inference
+
specialist inference
+
tool/search work
+
synthesis work
```

A subagent earns its place only if that extra work creates enough incremental
quality, coverage, reliability, or latency benefit.

# Notebook 13 — Key takeaways

1. Subagents are specialist workers, not primitive tools.
2. The parent acts like a research lead.
3. The strongest reason to use subagents is specialization + context partitioning.
4. Start with isolated specialists.
5. Descriptions matter for routing.
6. Skills encode procedures; subagents execute delegated work.
7. Simple tasks should stay with the parent.
8. Parallelism trades latency against resource use.
9. Main-context savings do not imply total-token savings.
10. Evaluate delegation separately from final quality.
11. Watch duplicate work and over-agenting.
12. Subagents must earn their complexity economically.

Next:
**Trace + Eval driven improvement**